## Install Dependencies 

In [ ]:
!pip install langgraph
!pip install langchain
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pandas
!pip install openai

In [ ]:
!pip install langchain-openai

In [3]:
import os 
os.chdir(r'D:/support_ticket_data')

## Load Dataset 

In [4]:
import pandas as pd

df = pd.read_csv("support_ticket_data.csv")

tickets = df["support_ticket_text"].tolist()

## Create Embeddings 

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(tickets)

 ## Vector Database (FAISS) 

In [ ]:
import faiss
import numpy as np # type: ignore
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "Your OpenAPI key here"              # Tested with my own API key .

## Instaniate the model 

In [21]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)

## Retrieval

In [ ]:
def retrieve(state):

    query = state["ticket"]

    query_embedding = model.encode([query])

    distances, indices = index.search(query_embedding, 3)

    retrieved = df.iloc[indices[0]]["support_ticket_text"].tolist()

    return {"context": retrieved}

## LLM Reasoning , Augmentation and Generation 

In [ ]:
def reasoning(state):

    ticket = state.get("ticket")
    context = "\n".join(state["context"])

    prompt = f"""
    Classify the support ticket.

    Categories:
    - Network Issue
    - Hardware Issue
    - Software Issue
    - Account Access Issue
    - Data Recovery

    Similar tickets:
    {context}

    Ticket:
    {ticket}

    Return:
    category and confidence score
    """

    response = llm.invoke(prompt)

    return {"decision": response}

## Workflow Control 

In [24]:
def decision_node(state):

    result = state["decision"]

    if result["confidence"] > 0.7:
        return "end"

    return "human_review"

## LangGraph Build

In [19]:
from langgraph.graph import StateGraph

builder = StateGraph(dict)

builder.add_node("retrieve", retrieve_node)

builder.add_node("reason", reasoning_node)

builder.set_entry_point("retrieve")

builder.add_edge("retrieve", "reason")

builder.add_conditional_edges(
    "reason",
    decision_node
)

graph = builder.compile()

## Run Langraph to Test 

In [ ]:
graph.invoke({
    "ticket": "My laptop keeps crashing and showing blue screen"
})